In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.ensemble import GradientBoostingRegressor

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
import xgboost
print(xgboost.__version__)

3.3.0


In [3]:
expenses = pd.read_csv("Expenses_Processed.csv")

In [4]:
expenses["date_time"] = pd.to_datetime(expenses["date_time"])

In [5]:
expenses["Year"] = expenses["date_time"].dt.year
expenses["Month"] = expenses["date_time"].dt.month
expenses["Day"] = expenses["date_time"].dt.day
expenses["Weekday"] = expenses["date_time"].dt.weekday

expenses["Weekend"] = (
    expenses["Weekday"] >= 5
).astype(int)

expenses["Quarter"] = expenses["date_time"].dt.quarter

In [11]:
from sklearn.preprocessing import LabelEncoder

category_encoder = LabelEncoder()
account_encoder = LabelEncoder()
currency_encoder = LabelEncoder()
tag_encoder = LabelEncoder()

expenses["category"] = category_encoder.fit_transform(expenses["category"])

expenses["account"] = account_encoder.fit_transform(expenses["account"])

expenses["currency"] = currency_encoder.fit_transform(expenses["currency"])

expenses["tags"] = tag_encoder.fit_transform(expenses["tags"])

In [12]:
features = [
    "category",
    "account",
    "currency",
    "tags",
    "Year",
    "Month",
    "Day",
    "Weekday",
    "Weekend",
    "Quarter"
]

X = expenses[features]

y = expenses["amount"]

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(750, 10)
(188, 10)


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [15]:
models = {

    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(random_state=42),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    )

}

In [20]:
results = []

best_model = None
best_score = -999

for name, model in models.items():

    model.fit(X_train, y_train)

    prediction = model.predict(X_test)

    mae = mean_absolute_error(y_test, prediction)

    mse = mean_squared_error(y_test, prediction)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, prediction)

    results.append([
        name,
        mae,
        mse,
        rmse,
        r2
    ])

    if r2 > best_score:

        best_score = r2

        best_model = model

In [21]:
result_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MAE",
        "MSE",
        "RMSE",
        "R2 Score"
    ]
)

result_df.sort_values(
    by="R2 Score",
    ascending=False
)

,Model,MAE,MSE,RMSE,R2 Score
0,Linear Regression,17.383110,779.442230,27.918493,-0.236448
3,Gradient Boosting,14.292973,1800.692101,42.434563,-1.856482
2,Random Forest,13.266290,2824.498963,53.146015,-3.480572
4,XGBoost,14.817486,4238.692874,65.105245,-5.723942
1,Decision Tree,14.968972,4864.583703,69.746568,-6.716808


In [22]:
print(best_model)


LinearRegression()


In [23]:
joblib.dump(best_model, "expense_model.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [24]:
if hasattr(best_model, "feature_importances_"):

    importance = pd.DataFrame({

        "Feature": features,

        "Importance": best_model.feature_importances_

    })

    importance = importance.sort_values(
        by="Importance",
        ascending=False
    )

    print(importance)

In [28]:
import matplotlib.pyplot as plt

if hasattr(best_model, "feature_importances_"):

    importance.plot(
        x="Feature",
        y="Importance",
        kind="bar",
        figsize=(10,5)
    )

    plt.title("Feature Importance")

    plt.show()

In [27]:
sample = X_test[0].reshape(1, -1)

prediction = best_model.predict(sample)

print("Predicted Expense:", prediction[0])

print("Actual Expense:", y_test.iloc[0])

Predicted Expense: 16.021904985470435
Actual Expense: 1.0


In [29]:
joblib.dump(category_encoder, "category_encoder.pkl")
joblib.dump(account_encoder, "account_encoder.pkl")
joblib.dump(currency_encoder, "currency_encoder.pkl")
joblib.dump(tag_encoder, "tag_encoder.pkl")

print("All Files Saved Successfully!")

All Files Saved Successfully!


In [30]:
pip install fastapi uvicorn



   ---------- ----------------------------- 1/4 [uvicorn]
   -------------------- ------------------- 2/4 [starlette]
   -------------------- ------------------- 2/4 [starlette]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ---------------------------------------- 4/4 [fastapi]

Note: you may need to restart the kernel to use updated packages.


In [31]:
!pip show uvicorn


Name: uvicorn
Version: 0.51.0
Summary: The lightning-fast ASGI server.
Home-page: https://uvicorn.dev/
Author: 
Author-email: Tom Christie <tom@tomchristie.com>
License-Expression: BSD-3-Clause
Location: C:\Users\LOQ\anaconda3\Lib\site-packages
Requires: click, h11
Required-by: 


In [32]:
import uvicorn
print(uvicorn.__version__)

0.51.0
